In [ ]:
# Custom SKU setup with explicit cost/price vectors
import numpy as np
from retail_world_model.envs.grocery import GroceryPricingEnv

# Define 5 SKUs with custom base prices and costs
base_prices = np.array([2.99, 1.49, 3.49, 0.99, 4.99])
base_costs = np.array([1.80, 0.90, 2.10, 0.60, 3.00])

env = GroceryPricingEnv(
    n_skus=5,
    episode_length=13,
    base_prices=base_prices,
    base_costs=base_costs,
)
print(f'Environment created with {env.n_skus} SKUs')
print(f'Base prices: {base_prices}')
print(f'Base margins: {(base_prices - base_costs) / base_prices * 100:.1f}%')

## Action Space

The action space is `Discrete(21)` per SKU, representing 21 price multipliers
from 0.90 to 1.10 in 1 percentage-point steps:

| Action | Multiplier | Meaning |
|--------|-----------|----------|
| 0      | 0.90      | -10% discount |
| 10     | 1.00      | No change |
| 20     | 1.10      | +10% markup |

For multi-SKU environments, the action is a vector of per-SKU discrete actions.

In [ ]:
# Step through the environment manually
obs, info = env.reset(seed=0)

# Keep prices flat for 5 weeks, then discount SKU 0
actions = [
    np.array([10, 10, 10, 10, 10]),  # week 1: no change
    np.array([10, 10, 10, 10, 10]),  # week 2: no change
    np.array([10, 10, 10, 10, 10]),  # week 3: no change
    np.array([10, 10, 10, 10, 10]),  # week 4: no change
    np.array([10, 10, 10, 10, 10]),  # week 5: no change
    np.array([ 5, 10, 10, 10, 10]),  # week 6: SKU 0 at -5%
    np.array([ 3, 10, 10, 10, 10]),  # week 7: SKU 0 at -7%
]

for i, action in enumerate(actions):
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'Week {i+1}: reward={reward:.3f}, prices={info.get("prices", "N/A")}')

In [ ]:
# Wrap with Stable Baselines 3 for baseline comparison
# (requires: pip install stable-baselines3)
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv

    vec_env = DummyVecEnv([lambda: GroceryPricingEnv(n_skus=3, episode_length=13)])
    model = PPO('MlpPolicy', vec_env, verbose=0, n_steps=64)
    model.learn(total_timesteps=1000)
    print('SB3 PPO baseline trained (1000 steps)')
except ImportError:
    print('stable-baselines3 not installed. Install with: pip install stable-baselines3')

In [ ]:
# Evaluation loop
import numpy as np

def evaluate(env, n_episodes=10, policy='random'):
    """Evaluate a policy over multiple episodes."""
    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        total_reward = 0.0
        done = False
        while not done:
            if policy == 'random':
                action = env.action_space.sample()
            elif policy == 'hold':
                action = np.full(env.n_skus, 10, dtype=int)  # no change
            else:
                action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        returns.append(total_reward)
    return np.array(returns)

eval_env = GroceryPricingEnv(n_skus=3, episode_length=13)
random_returns = evaluate(eval_env, policy='random')
hold_returns = evaluate(eval_env, policy='hold')

print(f'Random policy:  mean={random_returns.mean():.2f}, std={random_returns.std():.2f}')
print(f'Hold policy:    mean={hold_returns.mean():.2f}, std={hold_returns.std():.2f}')